# ETL — VISp Excitatory Patch-seq: DataSet & DataItem

Writes one `DataSet` record (`dataset_id = "visp_exc_patchseq"`, `project_id = "visp_patchseq"`), one `DataItem` per cell from `inferred_met_types.csv`, and the corresponding `DataItemDataSetAssociation` links. No prerequisites; features and cluster mappings are written in `_02` and `_03`.

In [1]:
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.arrow_utils import (
    build_arrow_schema,
    models_to_table,
    attach_linkml_metadata,
)
from connects_common_connectivity.models import (
    DataSet,
    DataItem,
    DataItemDataSetAssociation,
    Modality,
)
from connects_common_connectivity.write_utils import append_new_dataitems

In [2]:
INPUT_CSV   = "/data/visp-features-and-mapping/inferred_met_types.csv"
OUTPUT_ROOT = "../scratch/em_patchseq_wnm_v1/"
PROJECT_ID  = "visp_patchseq"
DATASET_ID  = "visp_exc_patchseq"

print(f"INPUT_CSV   : {INPUT_CSV}")
print(f"OUTPUT_ROOT : {OUTPUT_ROOT}")
print(f"PROJECT_ID  : {PROJECT_ID}")
print(f"DATASET_ID  : {DATASET_ID}")

INPUT_CSV   : /data/visp-features-and-mapping/inferred_met_types.csv
OUTPUT_ROOT : ../scratch/em_patchseq_wnm_v1/
PROJECT_ID  : visp_patchseq
DATASET_ID  : visp_exc_patchseq


## Load input CSV

In [3]:
df = pd.read_csv(INPUT_CSV, index_col=0)
df.index = df.index.astype(str)
print("Shape:", df.shape)
df.head(3)

Shape: (1528, 3)


,t_type,met_type,inferred_met_type
908902400,L6 CT VISp Ctxn3 Sla,NaN,L6 CT-1
965091329,L6 CT VISp Ctxn3 Sla,NaN,L6 CT-1
978149378,L5 ET VISp Krt80,NaN,L5 ET-2


## Write `DataSet`

In [4]:
dataset = DataSet(
    id=DATASET_ID,
    name="VISp excitatory Patch-seq data set",
    publication="doi.org/10.1101/2023.11.25.568393",
    modality=Modality.MORPHOLOGY.value,
    project_id=PROJECT_ID,
)

schema_ds = build_arrow_schema(DataSet)
table_ds  = models_to_table([dataset], schema=schema_ds)
table_ds  = attach_linkml_metadata(table_ds, linkml_class="DataSet")

# mode='overwrite' makes re-runs idempotent instead of appending duplicates.
# predicate scopes the overwrite to this project only — other projects' rows
# in the shared Delta table are left untouched.
write_deltalake(
    OUTPUT_ROOT + "dataset/",
    table_ds,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id"],
)
print("DataSet written:", table_ds.shape)

DataSet written: (1, 5)


In [5]:
# Verification
ds_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataset/")
    .filter(pl.col("project_id") == PROJECT_ID)
    .filter(pl.col("id") == DATASET_ID)
)
print(ds_verify.shape)
print(ds_verify.head())
assert ds_verify.shape[0] == 1, f"Expected 1 DataSet row, got {ds_verify.shape[0]}"
assert ds_verify["id"][0] == DATASET_ID, "DataSet id mismatch"

(1, 5)
shape: (1, 5)
┌───────────────────┬─────────────────┬───────────────────────────────┬────────────┬───────────────┐
│ id                ┆ name            ┆ publication                   ┆ modality   ┆ project_id    │
│ ---               ┆ ---             ┆ ---                           ┆ ---        ┆ ---           │
│ str               ┆ str             ┆ str                           ┆ str        ┆ str           │
╞═══════════════════╪═════════════════╪═══════════════════════════════╪════════════╪═══════════════╡
│ visp_exc_patchseq ┆ VISp excitatory ┆ doi.org/10.1101/2023.11.25.56 ┆ MORPHOLOGY ┆ visp_patchseq │
│                   ┆ Patch-seq data… ┆ 8…                            ┆            ┆               │
└───────────────────┴─────────────────┴───────────────────────────────┴────────────┴───────────────┘


## Write `DataItem`

In [6]:
cell_ids = df.index.tolist()

dataitems = [
    DataItem(id=cid, name=cid, project_id=PROJECT_ID)
    for cid in cell_ids
]

schema_di = build_arrow_schema(DataItem)
table_di  = models_to_table(dataitems, schema=schema_di)
table_di  = attach_linkml_metadata(table_di, linkml_class="DataItem")

# append_new_dataitems checks which ids already exist for this project and appends
# only new rows — safe when multiple _01 notebooks share a project_id, since
# each dataset's cells are registered without wiping the other's rows.
n_appended = append_new_dataitems(OUTPUT_ROOT + "dataitem/", table_di, project_id=PROJECT_ID)
print(f"DataItem rows appended: {n_appended} (total in batch: {len(cell_ids)})")

DataItem rows appended: 0 (total in batch: 1528)


In [7]:
# Verification
di_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataitem/")
    .filter(pl.col("project_id") == PROJECT_ID)
)
print(di_verify.shape)
print(di_verify.head())
registered_ids = set(di_verify["id"].to_list())
assert all(cid in registered_ids for cid in cell_ids), "Some cell_ids are missing from DataItem table"
assert di_verify["id"].n_unique() == di_verify.shape[0], "Duplicate DataItem ids detected"

(4407, 4)
shape: (5, 4)
┌───────────┬───────────┬───────────────────┬───────────────┐
│ id        ┆ name      ┆ neuroglancer_link ┆ project_id    │
│ ---       ┆ ---       ┆ ---               ┆ ---           │
│ str       ┆ str       ┆ str               ┆ str           │
╞═══════════╪═══════════╪═══════════════════╪═══════════════╡
│ 908902400 ┆ 908902400 ┆ null              ┆ visp_patchseq │
│ 965091329 ┆ 965091329 ┆ null              ┆ visp_patchseq │
│ 978149378 ┆ 978149378 ┆ null              ┆ visp_patchseq │
│ 834891776 ┆ 834891776 ┆ null              ┆ visp_patchseq │
│ 897003522 ┆ 897003522 ┆ null              ┆ visp_patchseq │
└───────────┴───────────┴───────────────────┴───────────────┘


## Write `DataItemDataSetAssociation`

In [8]:
associations = [
    DataItemDataSetAssociation(
        dataitem_id=cid,
        dataset_id=DATASET_ID,
        project_id=PROJECT_ID,
    )
    for cid in cell_ids
]

schema_assoc = build_arrow_schema(DataItemDataSetAssociation)
table_assoc  = models_to_table(associations, schema=schema_assoc)
table_assoc  = attach_linkml_metadata(table_assoc, linkml_class="DataItemDataSetAssociation")

# mode='overwrite' makes re-runs idempotent instead of appending duplicates.
# predicate scopes the overwrite to this project only — other projects' rows
# in the shared Delta table are left untouched.
write_deltalake(
    OUTPUT_ROOT + "dataitem_dataset_association/",
    table_assoc,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND dataset_id = '{DATASET_ID}'",
    partition_by=["project_id"],
)
print("DataItemDataSetAssociation written:", table_assoc.shape)

DataItemDataSetAssociation written: (1528, 3)


In [9]:
# Verification
assoc_verify = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
    .filter(pl.col("project_id") == PROJECT_ID)
    .filter(pl.col("dataset_id") == DATASET_ID)
)
print(assoc_verify.shape)
print(assoc_verify.head())
assert assoc_verify.shape[0] == len(cell_ids), (
    f"Expected {len(cell_ids)} association rows, got {assoc_verify.shape[0]}"
)
assert (assoc_verify["dataset_id"] == DATASET_ID).all(), "Not all associations point to DATASET_ID"

(1528, 3)
shape: (5, 3)
┌─────────────┬───────────────────┬───────────────┐
│ dataitem_id ┆ dataset_id        ┆ project_id    │
│ ---         ┆ ---               ┆ ---           │
│ str         ┆ str               ┆ str           │
╞═════════════╪═══════════════════╪═══════════════╡
│ 908902400   ┆ visp_exc_patchseq ┆ visp_patchseq │
│ 965091329   ┆ visp_exc_patchseq ┆ visp_patchseq │
│ 978149378   ┆ visp_exc_patchseq ┆ visp_patchseq │
│ 834891776   ┆ visp_exc_patchseq ┆ visp_patchseq │
│ 897003522   ┆ visp_exc_patchseq ┆ visp_patchseq │
└─────────────┴───────────────────┴───────────────┘


## Summary

| Output path | Class | Rows |
|---|---|---|
| `dataset/` | `DataSet` | 1 |
| `dataitem/` | `DataItem` | 1 528 |
| `dataitem_dataset_association/` | `DataItemDataSetAssociation` | 1 528 |

**Input columns intentionally not written here:**
- `t_type` — transcriptomic type label; written in a later notebook as `CellToClusterMapping`.
- `met_type`, `inferred_met_type` — MET-type labels; written in a later notebook as `ClusterMembership`.